# YouTube Comment Collection — FUZZ CHANNEL

**Project 2 | High Performance Data Processing (SECP3133)**  
**Data Source:** YouTube comments from all videos on FUZZ CHANNEL (Malaysian tech review channel)

---

This notebook collects raw YouTube comments from **all videos** published on **FUZZ CHANNEL**. The collected comments serve as the primary text corpus for downstream sentiment analysis.

### Pipeline overview

Youtube Data API v3 -> Channel ID -> Fetch Comments (Paginated) -> Save into **data/raw_data/youtube_comments_raw.csv**

### Collected fields
| Field | Description |
|---|---|
| `video_id` | YouTube video identifier |
| `video_title` | Title of the video |
| `video_published_at` | Upload date of the video |
| `comment_id` | Unique comment identifier |
| `author` | Display name of the commenter |
| `comment_text` | Raw comment text |
| `like_count` | Number of likes on the comment |
| `reply_count` | Number of replies to the comment thread |
| `published_at` | Timestamp the comment was posted |
| `updated_at` | Timestamp of last edit (if any) |

## 1. Setup and Imports

We rely on the following libraries:

- **`google-api-python-client`** — official Google client library wrapping the YouTube Data API v3.
- **`pandas`** — tabular data manipulation and CSV export.
- **`tqdm`** — lightweight progress bars for long-running loops.
- **`time` / `datetime`** — rate-limit back-off and timestamping.

> Install any missing package with: `pip install google-api-python-client pandas tqdm`

In [1]:
import os
import time
import json
from datetime import datetime

import pandas as pd
from tqdm import tqdm
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2. Configuration

Set your YouTube Data API v3 key and the collection parameters below.

| Parameter | Purpose |
|---|---|
| `API_KEY` | Authenticates requests to the YouTube Data API. |
| `CHANNEL_QUERY` | Search term used to locate FUZZ CHANNEL. |
| `MAX_COMMENTS_PER_VIDEO` | Upper bound on comments fetched per video (`None` = all). |
| `MAX_TOTAL_COMMENTS` | Hard cap on total comments collected across all videos. |
| `REQUEST_DELAY` | Seconds to wait between API calls to avoid quota bursts. |
| `OUTPUT_DIR` | Directory where the raw CSV will be written. |

> **Quota note:** The YouTube Data API v3 grants **10,000 units per day** on a free key.  
> Each `commentThreads.list` call costs **1 unit** and returns up to 100 comments.  
> With 10,000 units you can pull roughly 1,000,000 comments — more than enough for this project.

In [ ]:
# ── Authentication ─────────────────────────────────────────────────────────────
API_KEY = "YOUR_API_KEY_HERE"

# ── Channel targeting ──────────────────────────────────────────────────────────
CHANNEL_QUERY = "FUZZ CHANNEL"

# ── Collection limits ──────────────────────────────────────────────────────────
MAX_COMMENTS_PER_VIDEO = None
MAX_TOTAL_COMMENTS     = 12000
REQUEST_DELAY          = 0.3    # seconds between API calls

# ── Output ─────────────────────────────────────────────────────────────────────
OUTPUT_DIR  = r"c:\UTM things\Year 3\SEM 2\HDPD\Project 2\data\raw_data"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "youtube_comments_raw.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")

Output directory: c:\UTM things\Year 3\SEM 2\HDPD\Project 2\data\raw_data


## 3. Build the API Client

`googleapiclient.discovery.build` constructs a service object for the YouTube Data API v3.  
All subsequent calls go through this single client instance.


In [3]:
youtube = build("youtube", "v3", developerKey=API_KEY)
print("YouTube API client ready.")

YouTube API client ready.


## 4. Locate FUZZ CHANNEL

We use the **`search.list`** endpoint with `type=channel` to find FUZZ CHANNEL and retrieve its unique **channel ID**.  
The channel ID is then used to access the channel's upload playlist.

The **`channels.list`** endpoint returns a `contentDetails.relatedPlaylists.uploads` field — a special playlist that contains every video the channel has ever uploaded, in reverse-chronological order.

In [4]:
def get_channel_id(youtube_client, query):
    """
    Search for a YouTube channel by name and return (channel_id, channel_title).
    Raises ValueError if no channel is found.
    """
    response = youtube_client.search().list(
        q=query,
        type="channel",
        part="snippet",
        maxResults=5,
    ).execute()

    items = response.get("items", [])
    if not items:
        raise ValueError(f"No channel found for query: '{query}'")

    # Print all candidates so we can verify we picked the right one
    print("Candidate channels found:")
    for i, item in enumerate(items):
        cid   = item["snippet"]["channelId"]
        title = item["snippet"]["title"]
        print(f"  [{i}] {title}  (id: {cid})")

    channel_id    = items[0]["snippet"]["channelId"]
    channel_title = items[0]["snippet"]["title"]
    return channel_id, channel_title


def get_uploads_playlist_id(youtube_client, channel_id):
    """
    Return the uploads playlist ID for a given channel.
    This playlist contains all videos uploaded to the channel.
    """
    response = youtube_client.channels().list(
        id=channel_id,
        part="contentDetails",
    ).execute()

    return response["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]


# ── Execute ────────────────────────────────────────────────────────────────────
CHANNEL_ID, CHANNEL_TITLE = get_channel_id(youtube, CHANNEL_QUERY)
UPLOADS_PLAYLIST_ID       = get_uploads_playlist_id(youtube, CHANNEL_ID)

print(f"\nSelected channel : {CHANNEL_TITLE}")
print(f"Channel ID       : {CHANNEL_ID}")
print(f"Uploads playlist : {UPLOADS_PLAYLIST_ID}")

Candidate channels found:
  [0] FUZZ CHANNEL  (id: UCNgBXwadunmKRt0TksSFaqw)
  [1] Fuzz  (id: UCk_YaS7EjbnUe6r_1colvlA)
  [2] More Fuzz  (id: UC_I3PBtw880M77iQWKUY1Fg)
  [3] fuzz  (id: UCGacX4lqGSqYXZjnXrDPTOg)
  [4] Fakkah Fuzz  (id: UCMiaHiq_gGSe1vn-RqXZTog)

Selected channel : FUZZ CHANNEL
Channel ID       : UCNgBXwadunmKRt0TksSFaqw
Uploads playlist : UUNgBXwadunmKRt0TksSFaqw


## 5. Fetch All Uploaded Videos

The **`playlistItems.list`** endpoint is the most efficient way to enumerate a channel's videos.  
It returns up to **50 items per page**, so we paginate using `nextPageToken` until all pages are exhausted.

For each video we capture:
- `video_id` — needed to fetch its comments
- `video_title` — stored alongside each comment for reference and analysis
- `video_published_at` — for time-series analysis later

In [5]:
def fetch_all_videos(youtube_client, playlist_id):
    """
    Paginate through a playlist and return a list of video metadata dicts.
    Each dict contains: video_id, video_title, video_published_at.
    """
    videos     = []
    page_token = None

    print("Fetching video list from uploads playlist ...")
    while True:
        request_kwargs = dict(
            playlistId=playlist_id,
            part="snippet,contentDetails",
            maxResults=50,
        )
        if page_token:
            request_kwargs["pageToken"] = page_token

        response   = youtube_client.playlistItems().list(**request_kwargs).execute()
        page_items = response.get("items", [])

        for item in page_items:
            snippet = item["snippet"]
            videos.append({
                "video_id"           : snippet["resourceId"]["videoId"],
                "video_title"        : snippet.get("title", ""),
                "video_published_at" : snippet.get("publishedAt", ""),
            })

        page_token = response.get("nextPageToken")
        if not page_token:
            break

        time.sleep(REQUEST_DELAY)

    print(f"Total videos found: {len(videos)}")
    return videos


all_videos = fetch_all_videos(youtube, UPLOADS_PLAYLIST_ID)

Fetching video list from uploads playlist ...
Total videos found: 1248


## 6. Prepare Video List

All videos from the channel's uploads playlist are used. No filtering is applied — comments will be collected across the full range of FUZZ CHANNEL content, which covers tech reviews, tips, comparisons, and general tech news.

In [6]:
target_videos = all_videos

print(f"Total videos on channel : {len(all_videos)}")
print(f"Videos to collect from  : {len(target_videos)}")
print()
print("Sample videos:")
for v in target_videos[:10]:
    print(f"  [{v['video_published_at'][:10]}] {v['video_title']}")


Total videos on channel : 1248
Videos to collect from  : 1248

Sample videos:
  [2026-06-18] Selamat Tinggal iPad... Dah Jumpa Tablet Lagi Murah Yang Power!
  [2026-06-15] Brand Smartphone Yang Bukan Buat Phone Je! Tapi...
  [2026-06-11] Aku Beli iPhone 17 Pro Harga RM200! Tak Sangka…
  [2026-06-11] iPhone aku dah update iOS 27 😍
  [2026-06-11] iOS 27 vs iOS 26 : Clean Up 🔥
  [2026-06-09] iOS 27 dah rasmi dengan Siri AI 🎉
  [2026-06-09] Semua Ciri Baru iOS 27 Pengguna iPhone Wajib Tahu!
  [2026-06-05] HONOR Magic V6 Dah Boleh Connect iOS 27 Ecosystem!
  [2026-06-03] Social Media Kena Banned : 16 Tahun Ke Bawah Tak Boleh Guna! 🔥
  [2026-05-30] 25 Fungsi Menarik iPhone Tak Ramai Tahu!


## 7. Comment Collection — Helper Function

For each video we call **`commentThreads.list`**, which returns top-level comment threads.

### Pagination
The endpoint returns a maximum of **100 comments per page**. We loop on `nextPageToken` until either:
- There are no more pages, or
- We have reached `MAX_COMMENTS_PER_VIDEO` (if set).

### Error handling
| HTTP error | Cause | Action |
|---|---|---|
| 403 `commentsDisabled` | Channel/video has comments turned off | Skip video, log warning |
| 403 `forbidden` | Video is private or age-restricted | Skip video, log warning |
| 404 | Video deleted after playlist was fetched | Skip video, log warning |
| 429 / 5xx | Quota exhausted or server error | Raise immediately |

In [7]:
def fetch_comments_for_video(youtube_client, video_id, max_comments=None):
    """
    Fetch all top-level comments for a single video.

    Returns a list of comment dicts.
    Returns an empty list if comments are disabled or the video is unavailable.
    """
    comments   = []
    page_token = None

    while True:
        try:
            request_kwargs = dict(
                videoId=video_id,
                part="snippet",
                maxResults=100,
                textFormat="plainText",
                order="relevance",
            )
            if page_token:
                request_kwargs["pageToken"] = page_token

            response = youtube_client.commentThreads().list(**request_kwargs).execute()

        except HttpError as e:
            error_reason = ""
            try:
                error_body   = json.loads(e.content)
                error_reason = error_body["error"]["errors"][0]["reason"]
            except Exception:
                pass

            if e.resp.status in (403, 404):
                # Comments disabled, private video, or age-restricted — skip gracefully
                print(f"    [SKIP] {video_id}: {error_reason or e.resp.status}")
                return []
            else:
                raise  # Re-raise unexpected errors (quota exhausted, server errors)

        for item in response.get("items", []):
            top = item["snippet"]["topLevelComment"]["snippet"]
            comments.append({
                "comment_id"   : item["snippet"]["topLevelComment"]["id"],
                "author"       : top.get("authorDisplayName", ""),
                "comment_text" : top.get("textDisplay", ""),
                "like_count"   : top.get("likeCount", 0),
                "reply_count"  : item["snippet"].get("totalReplyCount", 0),
                "published_at" : top.get("publishedAt", ""),
                "updated_at"   : top.get("updatedAt", ""),
            })

        # Check early-stop condition
        if max_comments and len(comments) >= max_comments:
            comments = comments[:max_comments]
            break

        page_token = response.get("nextPageToken")
        if not page_token:
            break

        time.sleep(REQUEST_DELAY)

    return comments


print("Comment fetcher defined.")

Comment fetcher defined.


## 8. Run the Collection Loop

Iterate over all videos and accumulate comments into a single flat list, stopping once the `MAX_TOTAL_COMMENTS` cap is reached.  
A `tqdm` progress bar tracks per-video progress, and running totals are printed after each video.


In [8]:
all_records = []
skipped     = []

print(f"Starting collection for {len(target_videos)} videos (cap: {MAX_TOTAL_COMMENTS:,} total comments) ...\n")

for video in tqdm(target_videos, desc="Videos", unit="video"):
    if len(all_records) >= MAX_TOTAL_COMMENTS:
        print(f"\nReached {MAX_TOTAL_COMMENTS:,} comment cap — stopping early.")
        break

    vid_id    = video["video_id"]
    vid_title = video["video_title"]

    remaining = MAX_TOTAL_COMMENTS - len(all_records)

    comments = fetch_comments_for_video(
        youtube,
        video_id=vid_id,
        max_comments=remaining,
    )

    if not comments:
        skipped.append(vid_id)
        continue

    for comment in comments:
        all_records.append({
            "video_id"           : vid_id,
            "video_title"        : vid_title,
            "video_published_at" : video["video_published_at"],
            **comment,
        })

    print(
        f"  [OK] {vid_title[:65]!r}  ->  {len(comments):,} comments  "
        f"(total so far: {len(all_records):,})"
    )
    time.sleep(REQUEST_DELAY)

print(f"\nCollection complete.")
print(f"  Total comments collected : {len(all_records):,}")
print(f"  Videos skipped           : {len(skipped)}")

Starting collection for 1248 videos (cap: 12,000 total comments) ...



Videos:   0%|          | 0/1248 [00:00<?, ?video/s]

  [OK] 'Selamat Tinggal iPad... Dah Jumpa Tablet Lagi Murah Yang Power!'  ->  35 comments  (total so far: 35)


Videos:   0%|          | 1/1248 [00:00<08:23,  2.47video/s]

  [OK] 'Brand Smartphone Yang Bukan Buat Phone Je! Tapi...'  ->  14 comments  (total so far: 49)


...

  [OK] 'HONOR Magic V5 Review : Foldable Termurah di Malaysia Berharga RM'  ->  34 comments  (total so far: 11,981)


Videos:  19%|█▉        | 243/1248 [02:08<11:58,  1.40video/s]

  [OK] 'Perubahan Besar iPhone 17 Air / 17 Pro Harga Cecah RM9000?!'  ->  19 comments  (total so far: 12,000)


Videos:  20%|█▉        | 244/1248 [02:08<08:49,  1.90video/s]


Reached 12,000 comment cap — stopping early.

Collection complete.
  Total comments collected : 12,000
  Videos skipped           : 1


## 9. Inspect the Raw Dataset

Before saving, we do a quick sanity check:
- Confirm the schema looks correct.
- Check for obvious issues such as blank comment text or duplicate comment IDs.
- Review the per-video comment distribution.

In [9]:
df_raw = pd.DataFrame(all_records)

print("=== Shape ===")
print(f"  Rows    : {df_raw.shape[0]:,}")
print(f"  Columns : {df_raw.shape[1]}")

print("\n=== Column types ===")
print(df_raw.dtypes)

print("\n=== Missing values ===")
print(df_raw.isnull().sum())

print("\n=== Duplicate comment IDs ===")
dupes = df_raw.duplicated(subset="comment_id").sum()
print(f"  {dupes} duplicate(s) found")

print("\n=== Sample rows ===")
df_raw[["video_title", "author", "comment_text", "like_count", "published_at"]].head(5)


=== Shape ===
  Rows    : 12,000
  Columns : 10

=== Column types ===
video_id                str
video_title             str
video_published_at      str
comment_id              str
author                  str
comment_text            str
like_count            int64
reply_count           int64
published_at            str
updated_at              str
dtype: object

=== Missing values ===
video_id              0
video_title           0
video_published_at    0
comment_id            0
author                0
comment_text          0
like_count            0
reply_count           0
published_at          0
updated_at            0
dtype: int64

=== Duplicate comment IDs ===
  0 duplicate(s) found

=== Sample rows ===


,video_title,author,comment_text,like_count,published_at
0,Selamat Tinggal iPad... Dah Jumpa Tablet Lagi ...,@faizulirfan22,"For me, tablet/ipad tak boleh replace the lapt...",23,2026-06-18T17:56:46Z
1,Selamat Tinggal iPad... Dah Jumpa Tablet Lagi ...,@adzalanmdamin8154,tablet huawei da lama pintas ipad pun...4 tahu...,11,2026-06-19T05:47:30Z
2,Selamat Tinggal iPad... Dah Jumpa Tablet Lagi ...,@ayeemasrii,hopefully through update features je yg boleh ...,1,2026-06-18T20:05:41Z
3,Selamat Tinggal iPad... Dah Jumpa Tablet Lagi ...,@zamzuri13,"nak connectkan AI platform dengan OpenClaw tu,...",2,2026-06-18T14:03:34Z
4,Selamat Tinggal iPad... Dah Jumpa Tablet Lagi ...,@muhammadnazirulbakarudin7250,Kenapa susunan icon dia lebih banyak dari magi...,2,2026-06-19T02:12:49Z


In [10]:
# Per-video comment count distribution
video_stats = (
    df_raw.groupby("video_title")["comment_id"]
    .count()
    .rename("comment_count")
    .sort_values(ascending=False)
    .reset_index()
)

print(f"Videos with collected comments: {len(video_stats)}")
print()
print("Top 10 most-commented videos:")
print(video_stats.head(10).to_string(index=False))

print("\nComment count stats:")
print(video_stats["comment_count"].describe().to_string())

Videos with collected comments: 243

Top 10 most-commented videos:
                                                           video_title  comment_count
        Sebab Aku CANCEL Beli Perodua QV-E! – Realiti Sewa Bateri EV 🔥            608
                                   Teknologi Phone Yang GAGAL + PUPUS!            323
                              Seram betul drive Proton Saga MC3 2026 😰            312
                                  7 Jenama Phone Yang Stop Jual Phone!            294
              Lenjan Proton Saga MC3 2026 Naik Genting! Tak Sangka....            285
First Time Beli Proton Saga MC3 Harga RM50K! – Test Semua Benda Baru 🔥            273
                                           Masalah dengan EV charger 🥵            223
                             Masalah Sebenar Proton Saga MC3 2026 Aku!            217
                    Baru Sebulan, iPhone 17 Pro Max Dah Ada MASALAH! 😰            196
                         Unboxing iPhone 17 Pro Max : Semua Yang Baru!   

## 10. Save Raw Data

We save the unmodified DataFrame to `data/raw_data/youtube_comments_raw.csv`.

- **UTF-8-sig** encoding is used so the file opens correctly in Excel (adds a BOM marker).
- The index is excluded (`index=False`) to keep the CSV clean.
- A **metadata JSON sidecar** is written alongside the CSV to document when and how the data was collected — useful for reproducibility.


In [11]:
# Save CSV
df_raw.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"Raw CSV saved  -> {os.path.abspath(OUTPUT_FILE)}")
print(f"  Rows : {len(df_raw):,}")
print(f"  Size : {os.path.getsize(OUTPUT_FILE) / 1024:.1f} KB")

# Save collection metadata sidecar
metadata = {
    "collected_at"       : datetime.utcnow().isoformat() + "Z",
    "channel_title"      : CHANNEL_TITLE,
    "channel_id"         : CHANNEL_ID,
    "uploads_playlist"   : UPLOADS_PLAYLIST_ID,
    "total_channel_vids" : len(all_videos),
    "videos_targeted"    : len(target_videos),
    "videos_skipped"     : len(skipped),
    "total_comments"     : len(df_raw),
    "output_file"        : os.path.abspath(OUTPUT_FILE),
}

meta_file = OUTPUT_FILE.replace(".csv", "_metadata.json")
with open(meta_file, "w", encoding="utf-8") as fh:
    json.dump(metadata, fh, indent=2)

print(f"Metadata saved -> {os.path.abspath(meta_file)}")


Raw CSV saved  -> c:\UTM things\Year 3\SEM 2\HDPD\Project 2\data\raw_data\youtube_comments_raw.csv
  Rows : 12,000
  Size : 3102.7 KB
Metadata saved -> c:\UTM things\Year 3\SEM 2\HDPD\Project 2\data\raw_data\youtube_comments_raw_metadata.json


## 11. Collection Summary

The table below summarises what was collected in this run.


In [12]:
summary = pd.DataFrame([
    {"Metric": "Channel",                        "Value": CHANNEL_TITLE},
    {"Metric": "Total videos on channel",         "Value": str(len(all_videos))},
    {"Metric": "Videos targeted",                 "Value": str(len(target_videos))},
    {"Metric": "Videos with comments collected",  "Value": str(len(target_videos) - len(skipped))},
    {"Metric": "Videos skipped (no comments)",    "Value": str(len(skipped))},
    {"Metric": "Total comments collected",        "Value": f"{len(df_raw):,}"},
    {"Metric": "Unique videos in dataset",        "Value": str(df_raw['video_id'].nunique())},
    {"Metric": "Output file",                     "Value": OUTPUT_FILE},
])

summary.index = [""] * len(summary)
display(summary)

,Metric,Value
,Channel,FUZZ CHANNEL
,Total videos on channel,1248
,Videos targeted,1248
,Videos with comments collected,1247
,Videos skipped (no comments),1
,Total comments collected,"12,000"
,Unique videos in dataset,243
,Output file,c:\UTM things\Year 3\SEM 2\HDPD\Project 2\data...
